# Experiment 7: Text Preprocessing and Embedding Generation Using NLP Libraries

**Saravanan P**

**24BAD105**



In [1]:
# Setup: install/import libraries and download NLTK resources
%pip -q install gensim

import re
import random
import numpy as np
import nltk
import scipy.linalg

# Compatibility fix for some SciPy/Gensim combinations
if not hasattr(scipy.linalg, "triu"):
    scipy.linalg.triu = np.triu

from nltk.corpus import brown, stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
from gensim.models import Word2Vec

random.seed(42)
np.random.seed(42)

for resource in ["brown", "punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

print("Libraries imported and NLTK resources downloaded successfully.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 31.0 MB/s eta 0:00:00
Libraries imported and NLTK resources downloaded successfully.


In [6]:
# PART A: Dataset Preparation and Text Preprocessing

STUDENT_NAME = "Saravanan P"
ROLL_NUMBER = "24BAD105"

print("Experiment No: 7")
print("Student Name:", STUDENT_NAME)
print("Roll Number:", ROLL_NUMBER)

# Load the Brown Corpus using its News category
original_sentences = [
    " ".join(sentence)
    for sentence in brown.sents(categories="news")
]

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word.isalpha()]  # remove punctuation/numbers
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens

processed_records = [
    (sentence, preprocess_text(sentence))
    for sentence in original_sentences
]
processed_records = [
    (original, tokens)
    for original, tokens in processed_records
    if tokens
]
processed_sentences = [tokens for _, tokens in processed_records]

print("\nPART A: DATASET PREPARATION")
print("Original sentences:", len(original_sentences))
print("Processed sentences:", len(processed_sentences))

for i, (original, processed) in enumerate(processed_records[:5], start=1):
    print(f"\nSentence {i}")
    print("Original:", original)
    print("Preprocessed:", " ".join(processed))


Experiment No: 7
Student Name: Saravanan P
Roll Number: 24BAD105

PART A: DATASET PREPARATION
Original sentences: 4623
Processed sentences: 4590

Sentence 1
Original: The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place .
Preprocessed: fulton county grand jury said friday investigation atlanta recent primary election produced evidence irregularity took place

Sentence 2
Original: The jury further said in term-end presentments that the City Executive Committee , which had over-all charge of the election , `` deserves the praise and thanks of the City of Atlanta '' for the manner in which the election was conducted .
Preprocessed: jury said presentment city executive committee charge election deserves praise thanks city atlanta manner election conducted

Sentence 3
Original: The September-October term jury had been charged by Fulton Superior Court Judge Durwood Pye to investigate repor

In [3]:
# PART B: Vocabulary Generation and Word Frequency Analysis

all_tokens = [word for sentence in processed_sentences for word in sentence]
vocabulary = sorted(set(all_tokens))
freq_dist = FreqDist(all_tokens)

print("PART B: VOCABULARY ANALYSIS")
print("Total number of tokens:", len(all_tokens))
print("Vocabulary size:", len(vocabulary))

print("\nTop 10 Most Frequent Words:")
for word, frequency in freq_dist.most_common(10):
    print(f"{word}: {frequency}")

print("\nSample vocabulary:")
print(vocabulary[:30])

print("\nFrequency distribution sample:")
print(dict(freq_dist.most_common(10)))


PART B: VOCABULARY ANALYSIS
Total number of tokens: 47682
Vocabulary size: 9896

Top 10 Most Frequent Words:
said: 406
year: 260
would: 249
new: 241
state: 223
one: 222
last: 177
two: 174
first: 158
president: 155

Sample vocabulary:
['aaron', 'abandoned', 'abandonment', 'abatuno', 'abbey', 'abe', 'abel', 'abell', 'abide', 'ability', 'ablaze', 'able', 'abner', 'aboard', 'abolish', 'abolition', 'abortive', 'abra', 'abraham', 'abroad', 'abrupt', 'absence', 'absent', 'absolute', 'absolutely', 'absorb', 'absorbed', 'abstaining', 'abstention', 'abuse']

Frequency distribution sample:
{'said': 406, 'year': 260, 'would': 249, 'new': 241, 'state': 223, 'one': 222, 'last': 177, 'two': 174, 'first': 158, 'president': 155}


In [4]:
# PART C: Word Embedding Generation Using Word2Vec (Skip-gram)

model = Word2Vec(
    sentences=processed_sentences,
    vector_size=100,  # embedding dimension required by the lab manual
    window=5,         # context-window size required by the lab manual
    min_count=2,
    sg=1,              # 1 = Skip-gram architecture
    workers=4,
    seed=42,
    epochs=20
)

print("PART C: WORD2VEC MODEL")
print("Architecture: Skip-gram")
print("Embedding dimension:", model.wv.vector_size)
print("Context window:", model.window)
print("Vocabulary size in model:", len(model.wv))

selected_word = "government" if "government" in model.wv.key_to_index else model.wv.index_to_key[0]
vector = model.wv[selected_word]
print("\nSelected word:", selected_word)
print("Embedding vector:")
print(vector)
print("Vector dimensions:", vector.shape)

print("\nFive most similar words:")
for word, score in model.wv.most_similar(selected_word, topn=5):
    print(f"{word}: {score:.4f}")


PART C: WORD2VEC MODEL
Architecture: Skip-gram
Embedding dimension: 100
Context window: 5
Vocabulary size in model: 5164

Selected word: government
Embedding vector:
[ 0.05046611  0.0732144   0.19533378  0.30024034  0.11872555 -0.15611115
  0.28990147  0.467749    0.14329016 -0.7492099  -0.55911773 -0.15748271
 -0.09567203  0.4545974  -0.09631915 -0.5499646  -0.0622576  -0.2133248
 -0.5576647   0.3602975   0.20183617  0.3503166  -0.15824637  0.1183492
 -0.20018235  0.29129627 -0.19713663  0.19370398 -0.27440828 -0.26265532
 -0.23591618 -0.17855756 -0.1142807  -0.16435987 -0.07851839 -0.40290335
  0.29900965  0.13019975 -0.18945952 -0.3397246  -0.06843834 -0.2859291
  0.14016564 -0.53067744  0.49201503 -0.08304792  0.5447334   0.39710057
  0.54585505 -0.60246646  0.06230256 -0.09539042 -0.15995978  0.19718279
  0.05037895  0.02619131 -0.04644739  0.10040955  0.19200741 -0.14561576
  0.27458397 -0.1647855  -0.22968678 -0.1864761   0.15820226 -0.26047483
 -0.20066217 -0.21147399 -0.609165

In [5]:
# PART D: Word Similarity Analysis and Experiment Management

candidate_pairs = [
    ("government", "state"),
    ("president", "country"),
    ("people", "government")
]

# Keep selected word pairs only when both words occur in the trained vocabulary
similarity_results = []
for word1, word2 in candidate_pairs:
    if word1 in model.wv.key_to_index and word2 in model.wv.key_to_index:
        similarity_results.append((word1, word2, float(model.wv.similarity(word1, word2))))

# If any chosen words are missing, use frequent in-vocabulary words as a fallback
if len(similarity_results) < 3:
    from itertools import combinations
    already_used = {frozenset((w1, w2)) for w1, w2, _ in similarity_results}
    common_words = [
        word for word, _ in freq_dist.most_common(100)
        if word in model.wv.key_to_index and len(word) > 2
    ]
    for word1, word2 in combinations(common_words, 2):
        if frozenset((word1, word2)) not in already_used:
            score = float(model.wv.similarity(word1, word2))
            similarity_results.append((word1, word2, score))
            already_used.add(frozenset((word1, word2)))
        if len(similarity_results) >= 3:
            break

print("PART D: COSINE SIMILARITY SCORES")
for word1, word2, score in similarity_results[:3]:
    print(f"Similarity between '{word1}' and '{word2}': {score:.4f}")

if similarity_results:
    best_pair = max(similarity_results, key=lambda item: item[2])
    print("\nMost similar pair:")
    print(f"{best_pair[0]} and {best_pair[1]}")
    print(f"Cosine similarity: {best_pair[2]:.4f}")
else:
    print("Could not find enough valid word pairs for comparison.")

print("\nObservation:")
print("Words with higher cosine similarity have more similar vector directions based on the contexts learned by Word2Vec.")

# Optional: save trained model in the Colab runtime
model.save("experiment7_word2vec.model")
print("\nExperiment completed successfully. Model saved as experiment7_word2vec.model")


PART D: COSINE SIMILARITY SCORES
Similarity between 'government' and 'state': 0.4497
Similarity between 'president' and 'country': 0.3604
Similarity between 'people' and 'government': 0.3574

Most similar pair:
government and state
Cosine similarity: 0.4497

Observation:
Words with higher cosine similarity have more similar vector directions based on the contexts learned by Word2Vec.

Experiment completed successfully. Model saved as experiment7_word2vec.model
